# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [6]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [7]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['TXLDMMCZKR', 'XUPJTBVHFD'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[20, 24, 12,  4, 13, 13,  3, 26, 11, 18],
       [24, 21, 16, 10, 20,  2, 22,  8,  6,  4]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 18, 11, 26,  3, 13, 13,  4, 12, 24],
       [ 0,  4,  6,  8, 22,  2, 20, 10, 16, 21]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[18, 11, 26,  3, 13, 13,  4, 12, 24, 20],
       [ 4,  6,  8, 22,  2, 20, 10, 16, 21, 24]], dtype=int32)>)


# 建立sequence to sequence 模型

In [8]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.v_sz = 27
        self.embed = tf.keras.layers.Embedding(self.v_sz, 64)
        self.enc_cell = tf.keras.layers.SimpleRNNCell(128)
        self.dec_cell = tf.keras.layers.SimpleRNNCell(128)
        self.dense = tf.keras.layers.Dense(self.v_sz)
    
    def call(self, enc_ids, dec_ids):
        # 编码器
        enc_emb = self.embed(enc_ids)                           # (batch, enc_len, 64)
        enc_out, enc_state = tf.keras.layers.RNN(
            self.enc_cell, return_sequences=True, return_state=True)(enc_emb)
        
        # 解码器初始状态 = 编码器最终状态
        dec_state = enc_state
        
        # 静态序列长度（因为训练时 seqlen 固定，如 20）
        dec_len = dec_ids.shape[1]  # 这是静态整数，例如 20
        
        # 解码器输入嵌入（teacher forcing）
        dec_emb = self.embed(dec_ids)                           # (batch, dec_len, 64)
        
        # 手动循环解码（使用 Python range，在 @tf.function 中会被展开为图节点）
        outputs = []
        for t in range(dec_len):
            inp = dec_emb[:, t, :]                              # (batch, 64)
            out, dec_state = self.dec_cell(inp, dec_state)      # out: (batch, 128)
            outputs.append(out)
        
        dec_outputs = tf.stack(outputs, axis=1)                 # (batch, dec_len, 128)
        logits = self.dense(dec_outputs)                        # (batch, dec_len, 27)
        return logits
    
    def encode(self, enc_ids):
        enc_emb = self.embed(enc_ids)
        enc_out, enc_state = tf.keras.layers.RNN(
            self.enc_cell, return_sequences=True, return_state=True)(enc_emb)
        # 返回格式与预测时的 decode 函数兼容（列表形式）
        return [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state):
        x_emb = self.embed(x)                                   # (batch, 64)
        out, new_state = self.dec_cell(x_emb, state)            # out: (batch, 128)
        logits = self.dense(out)                                # (batch, 27)
        token = tf.argmax(logits, axis=-1)
        return token, new_state

# Loss函数以及训练逻辑

In [9]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(3000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [10]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3006463
step 500 : loss 1.4403617
step 1000 : loss 0.89145935
step 1500 : loss 0.63297015
step 2000 : loss 0.3968299
step 2500 : loss 0.3690753


<tf.Tensor: shape=(), dtype=float32, numpy=0.31184330582618713>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [11]:
def sequence_reversal():
    def decode(init_state, steps=10):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 10)
    state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1]), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('PJKWKWZWUH', 'HUWZWKWKJP'), ('KIDCCMQBLA', 'ALBQMCCDIK'), ('XJLJWHKVYZ', 'ZYVKHWJLJX'), ('QXGOJBQKYC', 'CYKQBJOGXQ'), ('DAIKZRKXIR', 'RIXKRZKIAD'), ('UGAIJMFQTN', 'NTQFMJIAGU'), ('GCYNHJIZVI', 'IVZIJHNYCG'), ('HTIPQGLOBJ', 'JBOLGQPITH'), ('QGIQYYFZEX', 'XEZFYYQIGQ'), ('ODKZSHZNSX', 'XSNZHSZKDO'), ('UOQBRMTAXU', 'UXATMRBQOU'), ('RABMLESYCQ', 'QCYSELMBAR'), ('CKNTFXCAVF', 'FVACXFTNKC'), ('NFMKUKDWDL', 'LDWDKUKMFN'), ('BZPIIGEKBK', 'KBKEGIIPZB'), ('LHLBFHZYQX', 'XQYZHFBLHL'), ('VJPISFENCN', 'NCNEFSIPJV'), ('WDVKLYIFJL', 'LJFIYLKVDW'), ('QQOYIFVKVG', 'GVKVFIYOQQ'), ('AFHTROOGXE', 'EXGOORTHFA'), ('JBQZCIZRTZ', 'ZTRZICZQBJ'), ('GMYBIAOCGA', 'AGCOAIBYMG'), ('VEMLEOXXQY', 'YQXXOELMEV'), ('MFLURGBDDC', 'CDDBGRULFM'), ('ZKCBTHNHJW', 'WJHNHTBCKZ'), ('SSDBSWHXTM', 'MTXHWSBDSS'), ('CCLWBZLTSC', 'CSTLZBWLCC